In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyBboxPatch, FancyArrowPatch
import matplotlib.patches as mpatches
from itertools import combinations
from scipy.spatial.distance import cdist
import marker_enrichment
from matplotlib.patches import Patch
import logomaker               # logomaker

In [ ]:
BASE = Path("/store24/project24/ladcol_012/XplainTDA/Datasets")
OUTPUT_PATH = Path("/work/project/ladcol_023/Thesis_Plots")
DATASETS = ["PBMC", "BMMCMultiOme", "HumanBrain"]
MODALITIES = ["GEX","Peak"]
METRICS = ["eucl","cknn"]
EMBEDDING = "PCA"
CELLTYPE_COL = "Celltype"

SUBSETS = [f"subset_{i}" for i in range(5)]
CLUSTER_COL = "clusterID"

DIMENSION_COLORS = {
    0: "#FD6BE1",  # H0 - connected components
    1: "#3849B7",  # H1 - loops
    2: "#4DA6FB",  # H2 - voids
}

DATASET_COLORS = {
    "PBMC": "#FD6BE1",
    "BMMCMultiOme": "#3849B7",
    "HumanBrain": "#4DA6FB",
}

LINESTYLE = {
    "GEX": "-",
    "ATAC": ":",
}

SCORE_COLUMNS = {"ami": "AMI"}

labels = {0: "H0", 1: "H1", 2: "H2"}
HOMOLOGY_DIMS = [0, 1, 2]
QUANTILE = 0.95
FONT=20
K_VALUES = [5, 15, 50, 75, 100, 250, 500, 750, 1000, 1250]

# Figures (by order of appearance)

## 1. Visualization Example Homology

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

# Left triangle (unfilled) - v1, v2, v3
v1, v2, v3 = (0, 0), (1, 0), (0.5, 1)
tri_left = Polygon([v1, v2, v3], closed=True, fill=False, edgecolor='black', linewidth=1.2)
ax.add_patch(tri_left)

# Right triangle (filled) - w1, w2, w3
w1, w2, w3 = (2, 0), (3, 0), (2.5, 1)
tri_right = Polygon([w1, w2, w3], closed=True, fill=True,
                     facecolor='#2568a9', edgecolor='#1a4269', linewidth=1.2)
ax.add_patch(tri_right)

# Connecting line between the two triangles
ax.plot([v2[0], w1[0]], [v2[1], w1[1]], color='black', linewidth=1.2)

# Vertex labels
offset = 0.08
ax.text(v1[0]-offset, v1[1]-offset, r'$v_1$', ha='right', va='top', fontsize=12, style='italic')
ax.text(v2[0]+offset, v2[1]-offset, r'$v_2$', ha='left', va='top', fontsize=12, style='italic')
ax.text(v3[0], v3[1]+offset, r'$v_3$', ha='center', va='bottom', fontsize=12, style='italic')

ax.text(w1[0]-offset, w1[1]-offset, r'$w_1$', ha='right', va='top', fontsize=12, style='italic')
ax.text(w2[0]+offset, w2[1]-offset, r'$w_2$', ha='left', va='top', fontsize=12, style='italic')
ax.text(w3[0], w3[1]+offset, r'$w_3$', ha='center', va='bottom', fontsize=12, style='italic')

# Vertex markers
for p in [v1, v2, v3, w1, w2, w3]:
    ax.plot(*p, 'ko', markersize=3)

ax.set_xlim(-0.3, 3.3)
ax.set_ylim(-0.3, 1.3)
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.savefig('simplicalcomplex.png', dpi=300, bbox_inches='tight')

## 2. Visualization cknn

In [ ]:
def sample_uniform_disk(center, radius, n_points, rng):
    """Uniformly sample n_points inside a disk (constant density within cluster)."""
    angles = rng.uniform(0, 2 * np.pi, n_points)
    radii = radius * np.sqrt(rng.uniform(0, 1, n_points))
    x = center[0] + radii * np.cos(angles)
    y = center[1] + radii * np.sin(angles)
    return np.column_stack([x, y])

def kth_neighbor_distance(dist_matrix, k=5):
    """For each point, distance to its k-th closest neighbor (excluding itself)."""
    sorted_dists = np.sort(dist_matrix, axis=1)
    return sorted_dists[:, k]

def scaled_distance_matrix(points, k=5):
    """
    Local-scaling normalization:
    d_scaled(x, y) = d(x, y) / sqrt(d(x, x_k) * d(y, y_k))
    """
    dist_matrix = cdist(points, points)
    d_k = kth_neighbor_distance(dist_matrix, k=k)
    scale = np.sqrt(np.outer(d_k, d_k))
    scale[scale == 0] = 1e-12
    return dist_matrix / scale

def farthest_pair_from_matrix(dist_matrix):
    """Find the pair (i, j) with the max distance in a precomputed distance matrix."""
    i, j = np.unravel_index(np.argmax(dist_matrix), dist_matrix.shape)
    return (i, j), dist_matrix[i, j]

CONNECTION_COLOR = "#0050A4"
POINT_COLOR = "black"

def plot_connections(points, dist_matrix, threshold, idx_a, idx_b, title, ax, panel_label):
    n = len(points)

    for i, j in combinations(range(n), 2):
        if dist_matrix[i, j] <= threshold:
            p1, p2 = points[i], points[j]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=CONNECTION_COLOR,
                    linewidth=1.0, alpha=0.8, zorder=1)

    ax.scatter(*points[idx_a].T, color=POINT_COLOR, alpha=0.8, zorder=2, s=25)
    ax.scatter(*points[idx_b].T, color=POINT_COLOR, alpha=0.8, zorder=2, s=25)

    ax.set_title(title)
    ax.axis("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Panel label (A / B) in top-left corner, in axes-fraction coordinates
    ax.text(0.03, 0.95, panel_label, transform=ax.transAxes,
            fontsize=16, fontweight="bold", va="top", ha="left", zorder=11)

rng = np.random.default_rng(seed=42)

n_per_cluster = 25

# Cluster A: denser (smaller radius -> more points per unit area)
cluster_a = sample_uniform_disk(center=(0, 0), radius=1.0, n_points=n_per_cluster, rng=rng)

# Cluster B: sparser (larger radius -> fewer points per unit area)
cluster_b = sample_uniform_disk(center=(6, 0), radius=2.5, n_points=n_per_cluster, rng=rng)

labels = np.array([0] * n_per_cluster + [1] * n_per_cluster)
points = np.vstack([cluster_a, cluster_b])

idx_a = np.arange(0, n_per_cluster)
idx_b = np.arange(n_per_cluster, 2 * n_per_cluster)

# --- Raw (unscaled) distances ---
raw_dists = cdist(points, points)
sub_a_raw = raw_dists[np.ix_(idx_a, idx_a)]
sub_b_raw = raw_dists[np.ix_(idx_b, idx_b)]
_, dist_a_raw = farthest_pair_from_matrix(sub_a_raw)
_, dist_b_raw = farthest_pair_from_matrix(sub_b_raw)
threshold_raw = max(dist_a_raw, dist_b_raw)
print(f"[Raw] Cluster A farthest: {dist_a_raw:.3f}, Cluster B farthest: {dist_b_raw:.3f}")
print(f"[Raw] Threshold (max of the two): {threshold_raw:.3f}")

# --- Scaled distances ---
k = 5
scaled_dists = scaled_distance_matrix(points, k=k)
sub_a_scaled = scaled_dists[np.ix_(idx_a, idx_a)]
sub_b_scaled = scaled_dists[np.ix_(idx_b, idx_b)]
_, dist_a_scaled = farthest_pair_from_matrix(sub_a_scaled)
_, dist_b_scaled = farthest_pair_from_matrix(sub_b_scaled)
threshold_scaled = max(dist_a_scaled, dist_b_scaled)
print(f"[Scaled] Cluster A farthest: {dist_a_scaled:.3f}, Cluster B farthest: {dist_b_scaled:.3f}")
print(f"[Scaled] Threshold (max of the two): {threshold_scaled:.3f}")

# --- Plot side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
plot_connections(points, raw_dists, threshold_raw, idx_a, idx_b,
                  "", axes[0], panel_label="A")
plot_connections(points, scaled_dists, threshold_scaled, idx_a, idx_b,
                  "", axes[1], panel_label="B")

fig.canvas.draw()  

left_bbox = axes[0].get_position()
right_bbox = axes[1].get_position()

arrow_y = (left_bbox.y0 + left_bbox.y1) / 2
arrow_start_x = left_bbox.x1 + 0.02
arrow_end_x = right_bbox.x0 - 0.02

arrow = FancyArrowPatch(
    (arrow_start_x, arrow_y),
    (arrow_end_x, arrow_y),
    transform=fig.transFigure,
    arrowstyle="-|>",
    mutation_scale=25,
    linewidth=2,
    color="black",
    zorder=10,
)
fig.add_artist(arrow)

plt.savefig("cknn_demo.png", dpi=300, bbox_inches='tight')

## 3. Normalized q-Bottleneck Distance

In [ ]:
FONT=30
QUANTILES = [0.5, 0.75, 0.95, 1]

x_positions = np.arange(len(K_VALUES))

fig = plt.figure(figsize=(24, 7 * len(QUANTILES)))
subfigs = fig.subfigures(len(QUANTILES), 1, hspace=0.2)

panel_letters = [chr(ord("A") + i) for i in range(len(QUANTILES))]
handles, legend_labels = None, None

for row_idx, QUANTILE in enumerate(QUANTILES):
    subfig = subfigs[row_idx] if len(QUANTILES) > 1 else subfigs
    subfig.suptitle(f"q = {QUANTILE}", fontsize=FONT + 10, fontweight="bold", y=1.0)

    axes = subfig.subplots(1, 3, sharey=True)
    subfig.subplots_adjust(top=0.82)

    for ax_idx, (ax, dim) in enumerate(zip(axes, HOMOLOGY_DIMS)):
        for dataset in DATASETS:
            for modality in MODALITIES:
                df = load_metrics(dataset, modality)
                if df.empty:
                    continue

                if dim == 0:
                    df_dim = df[(df["homology_dim"] == dim) & (df.index == "full")].copy()
                else:
                    df_dim = df[(df["homology_dim"] == dim) & (df.index == "k=1250")].copy()

                if df_dim.empty:
                    continue

                df_dim["k"] = df_dim["label_b"].str.extract(r"k=(\d+)")[0].astype(int)
                df_dim = df_dim.sort_values("k")
                df_dim = df_dim.set_index("k").reindex(K_VALUES).reset_index()

                linestyle = "-" if modality == "GEX" else ":"
                ax.plot(
                    x_positions,
                    df_dim["distance_by_diameter"],
                    linestyle=linestyle,
                    marker="o",
                    markersize=12,
                    linewidth=6,
                    color=DATASET_COLORS[dataset],
                    label=f"{dataset} ({'ATAC' if modality == 'Peak' else 'GEX'})"
                )

        ax.set_title(f"$H_{dim}$", fontsize=FONT + 10, fontweight="bold")
        ax.set_xticks(x_positions)
        ax.set_xticklabels([str(k) for k in K_VALUES], fontsize=FONT + 4, rotation=45, ha="right")
        ax.tick_params(axis="y", labelsize=FONT + 4)
        if ax_idx == 1:
            ax.set_xlabel("Number of nearest neighbors ($k$)", fontsize=FONT + 4)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

        # Only the first (H_0) panel gets the row's letter
        if ax_idx == 0:
            ax.text(
                -0.1, 1.15,
                panel_letters[row_idx],
                transform=ax.transAxes,
                fontsize=FONT + 8,
                fontweight="bold",
                color="black",
                va="top",
                ha="right"
            )

    axes[0].set_ylabel("Bottleneck Distance\n(Normalized)", fontsize=FONT + 6)

    if row_idx == 0:
        handles, legend_labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    legend_labels,
    fontsize=FONT + 2,
    title="Dataset / Modality",
    title_fontsize=FONT + 4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.14),
    ncol=3
)

plt.savefig(
    OUTPUT_PATH / "knn_summary_all_quantiles.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
FONT = 28
def load_pairwise_distances(dataset, modality, quantile):
    directory = stability_dir(dataset, modality)
    csv_path = directory / "pairwise_quantile_bottleneck.csv"
    if not csv_path.exists():
        print(f"Skipping (not found): {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    df = df[np.isclose(df["quantile"], quantile)].copy()
    df["pair"] = df.apply(lambda row: tuple(sorted([row["subset_a"], row["subset_b"]])), axis=1)
    df = df.drop_duplicates(subset=["homology_dim", "pair"])
    return df


QUANTILES = [0.5, 0.75, 0.95, 1]
n_rows, n_cols = 2, 2  # 2 quantiles per row

fig = plt.figure(figsize=(32, 7 * n_rows))
subfigs = fig.subfigures(n_rows, n_cols, wspace=0.09, hspace=0.15)

panel_letters = [chr(ord("A") + i) for i in range(len(QUANTILES))]
handles, legend_labels = None, None

for q_idx, quantile in enumerate(QUANTILES):
    row_idx, col_idx = divmod(q_idx, n_cols)
    subfig = subfigs[row_idx, col_idx]
    subfig.suptitle(f"q = {quantile}", fontsize=FONT + 6, fontweight="bold")

    axes = subfig.subplots(1, 2, sharey=True)

    for mod_idx, modality in enumerate(MODALITIES):
        ax = axes[mod_idx]

        for dim in HOMOLOGY_DIMS:
            for dataset_idx, dataset in enumerate(DATASETS):
                df = load_pairwise_distances(dataset, modality, quantile)
                if df is None:
                    continue
                values = df.loc[df["homology_dim"] == dim, "distance_by_diameter"].dropna().values
                if len(values) == 0:
                    continue
                rng = np.random.default_rng(seed=dataset_idx * 100 + dim)
                x = dataset_idx + rng.uniform(-0.12, 0.12, size=len(values))
                ax.scatter(x, values, color=colors[dim], s=100, alpha=1, edgecolors="none", label=labels[dim])

        ax.set_xticks(range(len(DATASETS)))
        ax.set_xticklabels(DATASETS, fontsize=FONT, rotation=20, ha="right")
        ax.tick_params(axis="y", labelsize=FONT)
        ax.set_title(
            "GEX" if modality == "GEX" else "ATAC",
            fontsize=FONT + 4,
            fontweight="bold"
        )
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        if row_idx == n_rows - 1:
            ax.set_xlabel("Dataset", fontsize=FONT+4)
        

        # Only the left (GEX) panel gets the block's letter
        if mod_idx == 0:
            ax.text(
                -0.1, 1.15,
                panel_letters[q_idx],
                transform=ax.transAxes,
                fontsize=FONT + 8,
                fontweight="bold",
                color="black",
                va="top",
                ha="right"
            )

    axes[0].set_ylabel("Bottleneck Distance\n(Normalized)", fontsize=FONT+4)

    if q_idx == 0:
        handles, legend_labels = axes[0].get_legend_handles_labels()

legend_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=colors[dim],
           markeredgecolor="none", markersize=14, label=labels[dim])
    for dim in HOMOLOGY_DIMS
]
fig.legend(
    handles=legend_handles,
    title="Dimension",
    title_fontsize=FONT,
    fontsize=FONT - 2,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3
)

plt.savefig(
    OUTPUT_PATH / "Subsampling_Summary_all_quantiles.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 4. k-Nearest Neighbors

In [ ]:
def knn_dir(dataset: str, modality: str) -> Path:
    return (
        BASE
        / dataset
        / "XplainTDA"
        / modality
        / "Stability"
        / "PCA"
        / "kNN"
    )


def load_metrics(dataset: str, modality: str) -> pd.DataFrame:

    path = knn_dir(dataset, modality) / "quantile_bottleneck_metrics.csv"

    if not path.exists():
        print(f"Not found: {path}")
        return pd.DataFrame()

    df = pd.read_csv(path, index_col=0)

    # Keep only q = 0.95
    df = df[
        np.isclose(df["quantile"], QUANTILE)
    ].copy()

    return df

fig, axes = plt.subplots(
    1,
    3,
    figsize=(24, 7),
    sharey=True
)

x_positions = np.arange(len(K_VALUES))

for ax, dim in zip(axes, HOMOLOGY_DIMS):

    for dataset in DATASETS:

        for modality in MODALITIES:

            df = load_metrics(
                dataset,
                modality
            )

            if df.empty:
                continue

            # H0: full vs k
            if dim == 0:
                df_dim = df[
                    (df["homology_dim"] == dim)
                    & (df.index == "full")
                ].copy()

            # H1/H2: k=1250 vs k
            else:
                df_dim = df[
                    (df["homology_dim"] == dim)
                    & (df.index == "k=1250")
                ].copy()

            if df_dim.empty:
                continue

            # Extract numerical k
            df_dim["k"] = (
                df_dim["label_b"]
                .str.extract(r"k=(\d+)")[0]
                .astype(int)
            )

            df_dim = df_dim.sort_values("k")

            # Make sure k values are in the correct order
            df_dim = df_dim.set_index("k").reindex(K_VALUES).reset_index()

            linestyle = "-" if modality == "GEX" else ":"

            ax.plot(
                x_positions,
                df_dim["distance_by_diameter"],
                linestyle=linestyle,
                marker="o",
                markersize=12,
                linewidth=6,
                color=DATASET_COLORS[dataset],
                label=f"{dataset} ({'ATAC' if modality == 'Peak' else 'GEX'})"
            )


    ax.set_title(
        f"$H_{dim}$",
        fontsize=FONT + 14,
        fontweight="bold"
    )

    ax.set_xticks(x_positions)

    ax.set_xticklabels(
        [str(k) for k in K_VALUES],
        fontsize=FONT+6,
        rotation=45,
        ha="right",
    )

    ax.tick_params(
        axis="y",
        labelsize=FONT+6
    )

    ax.set_xlabel(
        "Number of nearest neighbors ($k$)",
        fontsize=FONT+10
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )


axes[0].set_ylabel(
    "Bottleneck Distance\n(Normalized, q=0.95)",
    fontsize=FONT+10
)


handles, legend_labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    legend_labels,
    fontsize=FONT + 6,
    title="Dataset / Modality",
    title_fontsize=FONT + 6,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    ncol=3
)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "knn_summary.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Subsampling

In [ ]:
# Helper
def subset_path(dataset: str, modality: str, subset: int) -> Path:
    return (
        BASE
        / dataset
        / "XplainTDA"
        / modality
        / "Stability"
        / "PCA"
        / "Subsampling"
        / "subsets"
        / f"{subset}"
        / f"tda_{subset}.joblib"
    )
def plot_diagram_on_ax(ax, dgms, title):
    all_finite_deaths = []
    all_births = []

    for dim, dgm in enumerate(dgms):
        if dgm.size == 0:
            continue
        births = dgm[:, 0]
        deaths = dgm[:, 1]
        finite_mask = np.isfinite(deaths)
        all_finite_deaths.append(deaths[finite_mask])
        all_births.append(births)

    all_finite_deaths = (
        np.concatenate(all_finite_deaths) if all_finite_deaths else np.array([1.0])
    )
    all_births = np.concatenate(all_births) if all_births else np.array([1.0])
    max_val = max(all_finite_deaths.max(), all_births.max()) * 1.05

    # diagonal reference line
    ax.plot([0, max_val], [0, max_val], color="gray", linestyle="--", linewidth=1, zorder=0)

    for dim, dgm in enumerate(dgms):
        if dgm.size == 0:
            continue
        births = dgm[:, 0]
        deaths = dgm[:, 1]
        finite_mask = np.isfinite(deaths)

        ax.scatter(
            births[finite_mask],
            deaths[finite_mask],
            c=colors[dim],
            label=labels[dim],
            s=60,
            alpha=1,
            edgecolors="none",
        )

        inf_mask = ~finite_mask
        if inf_mask.any():
            ax.scatter(
                births[inf_mask],
                [max_val] * inf_mask.sum(),
                c=colors[dim],
                marker="^",
                s=90,
                edgecolors="black",
                linewidths=0.5,
                zorder=3,
            )

    ax.set_xlim(0, max_val)
    ax.set_ylim(0, max_val)
    ax.set_title(title, fontsize = 20)
    ax.set_aspect("equal")
    ax.tick_params(axis="both", labelsize=20)
 
    # compute ticks once and apply to both axes so x and y always match
    shared_ticks = plt.MaxNLocator(nbins=6).tick_values(0, max_val)
    shared_ticks = shared_ticks[(shared_ticks >= 0) & (shared_ticks <= max_val)]
    ax.set_xticks(shared_ticks)
    ax.set_yticks(shared_ticks)
 
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: "" if v == 0 else f"{v:g}")
    )

def plot_heatmaps(df: pd.DataFrame, axes, fig):

    FONT = 20

    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")

    all_vals = []
    matrices = {}

    for dim in HOMOLOGY_DIMS:
        mat = build_matrix(df, dim, QUANTILE)
        matrices[dim] = mat
        vals = mat.values[np.isfinite(mat.values)]
        if vals.size:
            all_vals.extend(vals)

    vmin = min(all_vals)
    vmax = max(all_vals)

    for ax, dim in zip(axes, HOMOLOGY_DIMS):

        mat = matrices[dim]
        masked = np.ma.masked_invalid(mat.values)

        im = ax.imshow(masked, cmap=cmap, vmin=vmin, vmax=vmax)

        tick_labels = [
            f"Subset {str(s).replace('subset_', '')}"
            for s in mat.columns
        ]

        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(
            tick_labels,
            rotation=45,
            ha="right",
            fontsize=FONT
        )

        if dim == 0:
            ax.set_yticks(range(len(mat.index)))
            ax.set_yticklabels(
                tick_labels,
                fontsize=FONT
            )
        else:
            ax.set_yticks([])
            ax.set_yticklabels([])

        ax.set_title(
            f"$H_{dim}$",
            fontsize=FONT+4,
            fontweight="bold"
        )

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if i == j:
                    continue

                val = mat.values[i, j]

                if not np.isfinite(val):
                    continue

                ax.text(
                    j,
                    i,
                    f"",
                    ha="center",
                    va="center",
                    color="white" if val > (vmin + vmax) / 2 else "white",
                    fontsize=16,
                )

    # Dedicated colorbar axis
    cbar_ax = fig.add_axes([0.8, 0.15, 0.015, 0.25])
    
    cbar = fig.colorbar(
        im,
        cax=cbar_ax
    )
    cbar.set_label(
        "Bottleneck Distance\n(Normalized, q=0.95)",
        fontsize=FONT
    )
    cbar.ax.tick_params(labelsize=FONT)
    cbar.locator = MaxNLocator(nbins=4)
    cbar.update_ticks()

def stability_dir(dataset: str, modality: str) -> Path:
    return (
        BASE
        / dataset
        / "XplainTDA"
        / modality
        / "Stability"
        / "PCA"
        / "Subsampling"
    )

def build_matrix(
    df: pd.DataFrame,
    dim: int,
    quantile: float
) -> pd.DataFrame:
    """Build a symmetric subset x subset matrix for one
    homology dimension and quantile."""

    sub = df[
        (df["homology_dim"] == dim)
        & (np.isclose(df["quantile"], quantile))
    ]

    mat = pd.DataFrame(
        np.nan,
        index=SUBSETS,
        columns=SUBSETS
    )

    for _, row in sub.iterrows():
        a = row["subset_a"]
        b = row["subset_b"]
        val = row["distance_by_diameter"]

        mat.loc[a, b] = val
        mat.loc[b, a] = val

    np.fill_diagonal(mat.values, np.nan)

    return mat


In [ ]:
for dataset in DATASETS:
    for modality in MODALITIES:

        directory = stability_dir(dataset, modality)
        csv_path = directory / "pairwise_quantile_bottleneck.csv"

        if not csv_path.exists():
            print(f"Skipping (not found): {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        # Create combined figure
        fig = plt.figure(figsize=(20, 9))
        
        gs = fig.add_gridspec(
            2, 5,
            height_ratios=[1, 1],
            hspace=0.45,
            wspace=0.35
        )
        
        # Top row
        diag_axes = [
            fig.add_subplot(gs[0, i])
            for i in range(5)
        ]
        
        # Bottom row uses the full width of the five columns
        bottom_gs = gs[1, :].subgridspec(
            1, 5,
            width_ratios=[0.5, 1, 1, 1, 0.5],
            wspace=0.35
        )
        
        heat_axes = [
            fig.add_subplot(bottom_gs[0, 1]),
            fig.add_subplot(bottom_gs[0, 2]),
            fig.add_subplot(bottom_gs[0, 3]),
        ]

        for i, subset in enumerate(SUBSETS):

            path = subset_path(dataset, modality, subset)
            data = joblib.load(path)
            dgms = data["dgms"]

            plot_diagram_on_ax(
                diag_axes[i],
                dgms,
                title=f"Subset {i}"
            )

            if i == 0:
                diag_axes[i].set_ylabel(
                    "Death [Filtration Value]",
                    fontsize=20
                )

        diag_axes[2].set_xlabel(
            "Birth [Filtration Value]",
            fontsize=20
        )

        # Legend
        handles, leg_labels = diag_axes[0].get_legend_handles_labels()

        fig.legend(
            handles,
            leg_labels,
            title="Dimension",
            title_fontsize=20,
            fontsize=18,
            loc="upper right",
            bbox_to_anchor=(0.99, 0.85)
        )


        # -----------------------
        # Second row: heatmaps
        # -----------------------

        plot_heatmaps(
            df=df,
            axes=heat_axes,
            fig=fig
        )
        if modality == "GEX":
            fig.text(
                0.03, 0.92, "A",
                fontsize=26,
                fontweight="bold",
                ha="center",
                va="center"
            )
            
            fig.text(
                0.03, 0.48, "B",
                fontsize=26,
                fontweight="bold",
                ha="center",
                va="center"
            )
        else:
            fig.text(
                0.03, 0.92, "C",
                fontsize=26,
                fontweight="bold",
                ha="center",
                va="center"
            )
            
            fig.text(
                0.03, 0.48, "D",
                fontsize=26,
                fontweight="bold",
                ha="center",
                va="center"
            )
            
        if modality == "Peak":
            run_mod = "ATAC"
        else:
            run_mod = "RNA"
        fig.suptitle(f"{dataset}-{run_mod}", fontsize= 24, fontweight="bold")
        
        fig.subplots_adjust(left=0.08)
        plt.savefig(f"/work/project/ladcol_023/Thesis_Plots/Subsampling_{dataset}_{modality}.png", dpi=300, bbox_inches='tight')
        
        plt.show()
        plt.close(fig)

In [ ]:
def load_pairwise_distances(dataset, modality):
    directory = stability_dir(dataset, modality)
    csv_path = directory / "pairwise_quantile_bottleneck.csv"

    if not csv_path.exists():
        print(f"Skipping (not found): {csv_path}")
        return None

    df = pd.read_csv(csv_path)

    # Keep only q = 0.95
    df = df[np.isclose(df["quantile"], QUANTILE)].copy()

    # Keep only unique subset pairs
    # This avoids plotting both (subset_0, subset_1)
    # and (subset_1, subset_0)
    df["pair"] = df.apply(
        lambda row: tuple(sorted([row["subset_a"], row["subset_b"]])),
        axis=1
    )

    df = df.drop_duplicates(
        subset=["homology_dim", "pair"]
    )

    return df


# ---------------------------------------------------------
# Create figure
# ---------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7),
    sharey=True
)

for ax, modality in zip(axes, MODALITIES):

    for dim in HOMOLOGY_DIMS:

        for dataset_idx, dataset in enumerate(DATASETS):

            df = load_pairwise_distances(
                dataset,
                modality
            )

            if df is None:
                continue

            values = df.loc[
                df["homology_dim"] == dim,
                "distance_by_diameter"
            ].dropna().values

            if len(values) == 0:
                continue

            # Small horizontal jitter so individual pairwise
            # comparisons can be seen
            rng = np.random.default_rng(
                seed=dataset_idx * 100 + dim
            )

            x = (
                dataset_idx
                + rng.uniform(
                    -0.12,
                    0.12,
                    size=len(values)
                )
            )

            ax.scatter(
                x,
                values,
                color=colors[dim],
                s=100,
                alpha=1,
                edgecolors="none",
                label=labels[dim]
            )

    # X-axis
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels(
        DATASETS,
        fontsize=FONT+4
    )

    ax.tick_params(
        axis="y",
        labelsize=FONT+4
    )

    ax.set_title(
        "GEX" if modality == "GEX" else "ATAC",
        fontsize=FONT + 8,
        fontweight="bold"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ax.set_xlabel(
        "Dataset",
        fontsize=FONT+4
    )

# Y-axis only on first plot
axes[0].set_ylabel(
    "Bottleneck Distance\n(Normalized, q=0.95)",
    fontsize=FONT+4
)

# One shared legend
handles, legend_labels = axes[0].get_legend_handles_labels()

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=colors[dim],
        markeredgecolor="none",
        markersize=14,
        label=labels[dim]
    )
    for dim in HOMOLOGY_DIMS
]

fig.legend(
    handles=legend_handles,
    title="Dimension",
    title_fontsize=FONT+2,
    fontsize=FONT+2,
    loc="lower center",
    bbox_to_anchor=(0.79, 0.75),
    ncol=3
)
plt.tight_layout()
plt.savefig(f"/work/project/ladcol_023/Thesis_Plots/Subsampling_Summary.png", dpi=300)
plt.show()

## 6. Cell Type Alignment with Connected Components + CkNN

In [ ]:

def combined_results_path(dataset, run_modality, embedding, metric):
    """
    combined_results.csv lives at
    OUT_PATH / dataset / run_modality / embedding / metric / "combined_results.csv"
    """
    return OUTPUT_PATH / dataset / run_modality / embedding / metric / "combined_results.csv"


def celltype_composition(dataset, run_modality, embedding, metric, celltype_col):
    """
    Reads combined_results.csv (index = barcodes, columns include
    celltype_col and CLUSTER_COL) and returns:
        composition: DataFrame, index=cluster id, columns=cell types, values=counts
    """
    path = combined_results_path(dataset, run_modality, embedding, metric)
    df = pd.read_csv(path, index_col=0)

    composition = (
        df
        .groupby([CLUSTER_COL, celltype_col])
        .size()
        .unstack(fill_value=0)
    )

    # order clusters largest -> smallest
    composition = composition.loc[
        composition.sum(axis=1).sort_values(ascending=False).index
    ]

    return composition, path


def build_color_map(celltypes):
    """
    Assigns each cell type a color from COLORS_TO_USE. If a dataset has
    more than 20 cell types, colors repeat.
    """
    n = len(colors_to_use)
    return {ct: colors_to_use[i % n] for i, ct in enumerate(celltypes)}


# ------------------------------------------------------------------
# PLOTTING
# ------------------------------------------------------------------
def plot_celltype_composition_all(embedding, metric, celltype_col, normalize=False):
    """
    One figure for a given embedding/metric combo, combining every
    dataset: rows = datasets, columns = GEX / ATAC. All panels are the
    same size (regular grid), share one title and one legend, and are
    labeled A, B, C, ... in bold black, paper style.
    """
    panel_data = {}

    for dataset in DATASETS:
        for run_modality in MODALITIES:
            composition, _ = celltype_composition(
                dataset, run_modality, embedding, metric, celltype_col
            )

            # drop clusters that only contain a single cell
            cluster_sizes = composition.sum(axis=1)
            composition = composition.loc[cluster_sizes > 1]

            if normalize:
                plot_df = composition.div(composition.sum(axis=1), axis=0)
            else:
                plot_df = composition

            # relabel cluster IDs as 1, 2, 3, ... (already ordered largest -> smallest)
            plot_df = plot_df.copy()
            plot_df.index = range(1, len(plot_df) + 1)

            panel_data[(dataset, run_modality)] = plot_df

    ylabel = "Proportion of cells" if normalize else "Number of cells"
    n_rows = len(DATASETS)
    n_cols = len(MODALITIES)

    # regular grid -> every panel is the same size
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 3.5 * n_rows))

    panel_letters = iter(string.ascii_uppercase)

    for row, dataset in enumerate(DATASETS):
        row_celltypes = sorted(set().union(
            *(panel_data[(dataset, m)].columns for m in MODALITIES)
        ))
        row_color_map = build_color_map(row_celltypes)

        for col, run_modality in enumerate(MODALITIES):
            ax = axes[row, col]
            plot_df = panel_data[(dataset, run_modality)]
            colors = [row_color_map[ct] for ct in plot_df.columns]
            plot_df.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.85, legend=False)

            ax.set_xlabel("Conneted Component ID", fontsize=11)
            if col == 0:
                ax.set_ylabel(f"$\\bf{{{dataset}}}$\n{ylabel}", fontsize=11)
            if row == 0:
                display_name = "ATAC" if run_modality == "Peak" else run_modality
                ax.set_title(display_name, fontsize=13)

            # thin out x-tick labels so they stay readable even when a
            # panel has many clusters — show at most ~15 labels
            n_bars = len(plot_df)
            step = max(1, n_bars // 15)
            tick_positions = list(range(0, n_bars, step))
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(
                [str(plot_df.index[i]) for i in tick_positions],
                rotation=90, fontsize=8,
            )

            # paper-style panel label (A, B, C, ...)
            ax.text(
                -0.15, 1.12, next(panel_letters),
                transform=ax.transAxes,
                fontsize=15, fontweight="bold", color="black",
                va="top", ha="left",
            )

        # one legend per row (dataset), attached to the rightmost panel,
        # showing only the cell types present in that dataset
        row_handles = [plt.Rectangle((0, 0), 1, 1, color=row_color_map[ct]) for ct in row_celltypes]
        axes[row, -1].legend(
            row_handles, row_celltypes, title="Cell Type",
            bbox_to_anchor=(1.02, 1), loc="upper left",
            fontsize=10, title_fontsize=11,
        )

    #fig.suptitle(
    #    f"Cell Type Composition per Connected Component",
    #    x=0.4, fontsize=17, fontweight="bold",
    #)

    fig.tight_layout(rect=[0, 0, 0.8, 0.96])

    # figure combines every dataset, so save at the top level
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    suffix = "proportion" if normalize else "counts"
    out_path = OUTPUT_PATH / f"celltype_composition_{suffix}_ALL_{embedding}_{metric}.pdf"
    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    print(f"Saved: {out_path}")

    # plt.show()
    plt.close(fig)


# ------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------
for metric in METRICS:
    for normalize in (False, True):
        try:
            plot_celltype_composition_all(
                EMBEDDING, metric, CELLTYPE_COL, normalize=normalize,
            )
        except Exception as e:
            print(f"Skipping {metric} | normalize={normalize}")
            print(f"Reason: {e}")

In [ ]:
# ------------------------------------------------------------------
# COLORS / STYLES
# ------------------------------------------------------------------

DATASET_COLORS = {
    "PBMC": "#FD6BE1",
    "BMMCMultiOme": "#3849B7",
    "HumanBrain": "#4DA6FB",
}

LINESTYLE = {
    "GEX": "-",
    "ATAC": ":",
}


# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

def load_curve(dataset, run_modality, embedding, metric):
    """
    Load ami_filtration_values.csv.

    Returns dataframe with:
        filtration_value
        n_components
        ami
    """

    path = ami_filtration_path(
        dataset,
        run_modality,
        embedding,
        metric,
    )

    df = pd.read_csv(path, index_col=0)

    df.index.name = "filtration_value"
    df = df.reset_index()

    required = {
        "filtration_value",
        "n_components",
        "ami",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path} is missing columns: {missing}"
        )

    df["filtration_value"] = pd.to_numeric(
        df["filtration_value"]
    )

    df["n_components"] = pd.to_numeric(
        df["n_components"]
    )

    df["ami"] = pd.to_numeric(
        df["ami"]
    )

    return df



# ------------------------------------------------------------------
# PLOT
# ------------------------------------------------------------------

def plot_all_datasets(all_curves, score_col, save_dir):

    score_label = SCORE_COLUMNS[score_col]

    plt.rcParams.update({
        "axes.labelsize": 26,
        "axes.titlesize": 26,
        "xtick.labelsize": 26,
        "ytick.labelsize": 26,
        "legend.fontsize": 26,
        "figure.dpi": 300,
    })


    fig, axes = plt.subplots(
        1,
        3,
        figsize=(22, 8),
        constrained_layout=True,
    )

    ax_cc, ax_fil, ax_betti = axes


    handles = []
    labels = []


    for dataset, curves in all_curves.items():

        color = DATASET_COLORS[dataset]


        for (modality, embedding, metric), df in curves.items():


            if embedding != "PCA":
                continue

            if metric != "eucl":
                continue


            # --------------------------------------------------
            # Cut filtration values at 75
            # --------------------------------------------------

            df_plot = df[
                df["filtration_value"] <= 25
            ].copy()


            if df_plot.empty:
                continue


            # --------------------------------------------------
            # Convert to numpy
            # --------------------------------------------------

            x_cc = df_plot[
                "n_components"
            ].to_numpy(dtype=float)

            x_fil = df_plot[
                "filtration_value"
            ].to_numpy(dtype=float)

            y = df_plot[
                score_col
            ].to_numpy(dtype=float)


            # Best AMI point

            best_idx = np.argmax(y)

            best_cc = x_cc[best_idx]
            best_fil = x_fil[best_idx]
            best_score = y[best_idx]


            style = {
                "color": color,
                "linestyle": LINESTYLE[modality],
                "linewidth": 2.5,
            }


            label = f"{dataset} ({modality})"


            # --------------------------------------------------
            # AMI vs connected components
            # --------------------------------------------------

            line, = ax_cc.plot(
                x_cc,
                y,
                **style,
            )

            ax_cc.scatter(
                best_cc,
                best_score,
                color="red",
                s=40,
                zorder=5,
            )


            # --------------------------------------------------
            # AMI vs filtration value
            # --------------------------------------------------

            ax_fil.plot(
                x_fil,
                y,
                **style,
            )

            ax_fil.scatter(
                best_fil,
                best_score,
                color="red",
                s=40,
                zorder=5,
            )


            # --------------------------------------------------
            # Betti curve
            # --------------------------------------------------

            ax_betti.plot(
                x_fil,
                x_cc,
                **style,
            )

            ax_betti.scatter(
                best_fil,
                best_cc,
                color="red",
                s=40,
                zorder=5,
            )


            handles.append(line)
            labels.append(label)



    # Remove duplicate legend entries

    unique = dict(zip(labels, handles))


    # Axis formatting

    ax_cc.set_xlabel(
        r"Number of Connected Components ($\beta_0$)"
    )
    ax_cc.set_ylabel(score_label)
    ax_cc.set_title(
        "Connected Components vs AMI",
        fontweight="bold",
    )
    ax_cc.grid(alpha=0.3)


    ax_fil.set_xlabel(
        "Filtration Value"
    )
    ax_fil.set_ylabel(score_label)
    ax_fil.set_title(
        "Filtration Value vs AMI",
        fontweight="bold",
    )
    ax_fil.grid(alpha=0.3)


    ax_betti.set_xlabel(
        "Filtration Value"
    )
    ax_betti.set_ylabel(
        r"Number of Connected Components ($\beta_0$)"
    )
    ax_betti.set_title(
        r"Betti Curve ($\beta_0$)",
        fontweight= "bold",
    )
    ax_betti.grid(alpha=0.3)

    # Add panel labels
    ax_cc.text(
        -0.1, 1.05, "A",
        transform=ax_cc.transAxes,
        fontsize=30,
        fontweight="bold",
        va="top",
    )

    ax_fil.text(
        -0.1, 1.05, "B",
        transform=ax_fil.transAxes,
        fontsize=30,
        fontweight="bold",
        va="top",
    )

    ax_betti.text(
        -0.1, 1.05, "C",
        transform=ax_betti.transAxes,
        fontsize=30,
        fontweight="bold",
        va="top",
    )



    fig.legend(
        unique.values(),
        unique.keys(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        frameon=True,
    )


    save_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out_path = save_dir / f"{score_col}_combined.pdf"

    fig.savefig(
        out_path,
        bbox_inches="tight",
        dpi=300,
    )

    print(f"Saved: {out_path}")

    plt.show()
    plt.close(fig)

all_curves = {}

for dataset in DATASETS:

    print(f"\nProcessing {dataset}")
    curves = {}
    for folder_modality, plot_modality in [
        ("GEX", "GEX"),
        ("Peak", "ATAC"),
    ]:
        try:
            df = load_curve(
                dataset,
                folder_modality,
                "PCA",
                "eucl",
            )
            curves[
                (plot_modality, "PCA", "eucl")
            ] = df

            print(
                f"Loaded {dataset} | {plot_modality}"
            )

        except Exception as e:
            print(
                f"Skipping {dataset} | {plot_modality}"
            )
            print(e)

    if curves:
        all_curves[dataset] = curves

print("\nLoaded datasets:")
for dataset, curves in all_curves.items():
    print(dataset, list(curves.keys()))

plot_all_datasets(
    all_curves,
    "ami",
    OUTPUT_PATH,
)

## 8. Gene-Wise Perturbation

In [ ]:
human_brain = {"Inhibitory_neuron": [
        "OTX2", "DMBX1", "SST", "TFAP2B", "LHX6", "CXCL14", "CASR",
        "MYO5B", "ANO1", "EYA4", "VIP", "HAPLN1", "LRRC38", "PTHLH",
        "TFAP2A", "SOX14", "SCGN", "TAC3", "CHRDL1", "SP8", "DCN",
        "WNT16", "TCIM", "SFRP1", "LIPG", "ADGRG6", "BHLHE22",
        "PAX2", "RGS5", "NMU", "C1QL1", "PCDH18", "ELFN1", "KMO",
        "PRDM1", "MCUB", "CRH", "CRHBP", "PAWR", "TNFSF15",
        "GATA3", "SLC6A5", "EDNRA", "HTR3A", "RNF144B", "SCARA5",
        "NDNF", "NPR3", "IQGAP2", "MSR1", "LMCD1", "PDGFD",
        "RELN", "NPPC", "SLC22A3", "CHST9", "GRPR", "COL15A1",
        "NPSR1", "OSTN", "PTGFR", "TAC1", "HGF", "SLC17A8",
        "FAM89A", "KCNJ5", "ITGA1", "PNOC", "KCNS3", "CPED1", "NGF",
        "CMTM8", "ANGPT1", "MYB", "NPY", "EREG", "PROM1", "MEPE",
        "IL18R1", "HPGD", "SHISA8", "MYH11", "CASP4LP", "DDR2",
        "SULF1", "SCTR", "HGFAC", "ADAMTS17", "MYBPC1", "NOG",
        "TMEM255A", "CYP26A1", "KIT", "NR2E1", "CLCNKA", "FBN2",
        "TSPAN12", "PDCL2", "XDH", "MAFB"
    ],

    "Astrocyte": [
        "FGFR3", "GFAP", "TNC", "SLC14A1", "GJA1", "ETNPPL", "CD38",
        "CD44", "ID3", "GLI3", "AQP4", "F3", "EFEMP1", "LGR6", "SDC4",
        "ANGPTL4", "OAF", "PAPLN", "GJB6", "PAX3", "ITPRID1",
        "NAA11", "LCAT", "ZNF98", "CPAMD8", "MLC1", "HPSE2",
        "SLC7A10", "AGT", "RASL12", "SMTN", "MT1G", "AEBP1", "EDNRB",
        "EMX2", "PYGM", "RANBP3L"
    ],

    "Oligodendrocyte": [
        "OPALIN", "CD9", "MAG", "CNDP1", "RNASE1", "MAL",
        "SLCO1A2", "FOLH1", "NKX6-2", "PLLP", "MBOAT1", "PTCSC3",
        "SLC5A11", "ITGA2", "S100A1", "MUSK", "MOG", "ASPA",
        "CARNS1", "PPP1R14A", "LRP2", "S100B", "SH3TC2", "SPATA22",
        "TMEM98"
    ],

    "Microglia": [
        "CD74", "LNCAROD", "HLA-DRA", "CX3CR1", "FYB1", "C3",
        "CSF1R", "DOCK8", "SYK", "SRGN", "RGS1", "ITGAX", "C1QC",
        "PTPRC", "AIF1", "CCDC26", "TTR", "GPR183", "CD69",
        "HLA-DRB1", "C1QB", "OLR1", "CD84", "HLA-DPA1", "TNFRSF1B",
        "TYROBP", "IFI30", "MS4A6A", "PIK3R5", "CXCR4", "HLA-DQA1",
        "HLA-DRB5"
    ],

    "Molec_Layer_Interneur": [
        "GAD1", "GAD2", "SLC32A1", "SLC6A1", "SORCS3", "NXPH1",
        "GJD2", "PVALB"
    ],
    
    "Purkinje_neuron": [
    "PCP2", "TRABD2B", "ZNF385C", "ATP2A3", "IL20RA",
    "GNG13", "CA8", "CALB1", "ITPR1", "FOXP2"
],
}

pbmc={'Monocytes': ['FCN1', 'FCNM', 'CD14', 'TCF7L2', 'TCF4', 'TCF-4', 'FCGR3A', 'CD16A', 'CD16', 'FCGR3', 'FCG3', 'IGFR3', 'IMD20', 'CD16-II', 'FCRIIIA', 'FcGRIIIA', 'LYN', 'JTK8', 'SAIDV', 'p53Lyn', 'p56Lyn'], 
 'DC': ['CLEC9A', 'CADM1', 'CLEC10A', 'FCER1A', 'CST3', 'COTL1', 'LYZ', 'DMXL2', 'GZMB', 'IL3RA', 'COBLL1', 'TCF4'],
 'NK': ['GNLY', 'NKG7', 'CD247', 'GRIK4', 'FCER1G', 'TYROBP', 'KLRG1', 'FCGR3A', 'ID2', 'PLCG2', 'SYNE1'], 
 'B': ['MS4A1', 'IL4R', 'IGHD', 'FCRL1', 'IGHM', 'SSPN', 'ITGB1', 'EPHA4', 'COL4A4', 'PRDM1', 'ZNF215', 'IRF4', 'CD38', 'XBP1', 'PAX5', 'BCL11A', 'BLK', 'MME', 'CD24', 'ACSM3', 'MSI2', 'MZB1', 'HSP90B1', 'FNDC3B', 'IGKC', 'JCHAIN', 'RF4'], 
 'T': ['CD4', 'IL7R', 'TRBC2', 'ITGB1', 'CCR7', 'CD8A', 'CD8B', 'GZMB', 'GZMA', 'CCL5', 'GZMK', 'GZMH', 'CD69', 'CD38', 'LEF1', 'TCF7']}


bmmcmulitome={'Monocyte': ['FCN1', 'CD14', 'TCF7L2', 'FCGR3A', 'LYN'], 
 'Progenitor': ['CD14', 'ID2', 'VCAN', 'S100A9', 'CLEC12A', 'KLF4', 'PLAUR', 'IGLL1', 'VPREB1', 'MME', 'EBF1', 'SSBP2', 'BACH2', 'CD79B', 'IGHM', 'PAX5', 'PRKCE', 'DNTT', 'MPO', 'BCL2', 'KCNQ5', 'CSF3R', 'PRTN3', 'NRIP1', 'MECOM', 'PROM1', 'CD34', 'NKAIN2', 'ZNF385D', 'ITGA2B', 'RYR3', 'PLCB1'], 
 'DC': ['CLEC9A', 'CADM1', 'CLEC10A', 'FCER1A', 'CST3', 'COTL1', 'LYZ', 'DMXL2', 'GZMB', 'IL3RA', 'COBLL1', 'TCF4'],
 'Erythroid': ['SLC4A1', 'SLC25A37', 'HBB', 'HBA2', 'HBA1', 'TFRC', 'MKI67', 'CDK6', 'SYNGR1', 'HBM', 'GYPA'], 
 'NK_ILC': ['GNLY', 'NKG7', 'CD247', 'GRIK4', 'FCER1G', 'TYROBP', 'KLRG1', 'FCGR3A', 'ID2', 'PLCG2', 'SYNE1'], 
 'B': ['MS4A1', 'IL4R', 'IGHD', 'FCRL1', 'IGHM', 'SSPN', 'ITGB1', 'EPHA4', 'COL4A4', 'PRDM1', 'ZNF215', 'IRF4', 'CD38', 'XBP1', 'PAX5', 'BCL11A', 'BLK', 'MME', 'CD24', 'ACSM3', 'MSI2', 'MZB1', 'HSP90B1', 'FNDC3B', 'IGKC', 'JCHAIN', 'RF4'], 
 'T': ['CD4', 'IL7R', 'TRBC2', 'ITGB1', 'CCR7', 'CD8A', 'CD8B', 'GZMB', 'GZMA', 'CCL5', 'GZMK', 'GZMH', 'CD69', 'CD38', 'LEF1', 'TCF7']}

In [ ]:
dataset_results = marker_enrichment.run_marker_enrichment()

In [ ]:
TITLE_FONTSIZE = 26
OTHER_FONTSIZE = 22
STAR_FONTSIZE = 20
EXCLUDE_GROUPS = {"Oligodendrocyte"}
bracket_y = 0.8
bracket_height = 0.02
star_y = bracket_y + bracket_height * 1.8
panel_labels = ["A", "B", "C"]

n_rows = len(DATASETS)
n_cols = len(HOMOLOGY_DIMS)
mid_col = n_cols // 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows), sharey='row')

for row_idx, dataset in enumerate(DATASETS):
    df_d = dataset_data[dataset]
    results_d = dataset_results[dataset]
    marker_dict = MARKER_SETS.get(dataset, {})
    groups = sorted(g for g in df_d['group'].unique() if g not in EXCLUDE_GROUPS)

    for col_idx, h_dim in enumerate(HOMOLOGY_DIMS):
        ax = axes[row_idx, col_idx] if n_rows > 1 else axes[col_idx]
        df_h = df_d[df_d['homology_dim'] == h_dim]

        group_centers = np.arange(1, len(groups) + 1) * 3
        marker_positions = group_centers - 0.5
        other_positions = group_centers + 0.5

        other_data, marker_data = [], []
        for group in groups:
            sub = df_h[df_h['group'] == group]
            markers = set(marker_dict.get(group, []))
            is_marker = sub['gene'].isin(markers)

            other_scores = sub.loc[~is_marker, 'score'].values
            marker_scores = sub.loc[is_marker, 'score'].values

            other_data.append(other_scores if len(other_scores) > 0 else np.array([0]))
            marker_data.append(marker_scores if len(marker_scores) > 0 else np.array([0]))

        parts_other = ax.violinplot(other_data, positions=other_positions, widths=0.9,
                                     showmedians=True, showextrema=True)
        parts_marker = ax.violinplot(marker_data, positions=marker_positions, widths=0.9,
                                      showmedians=True, showextrema=True)

        for pc in parts_other['bodies']:
            pc.set_facecolor('#FD6BE1')
            pc.set_alpha(0.7)
        for pc in parts_marker['bodies']:
            pc.set_facecolor('#3849B7')
            pc.set_alpha(0.7)
        for parts in (parts_other, parts_marker):
            parts['cmedians'].set_color('black')
            parts['cmaxes'].set_color('black')
            parts['cmins'].set_color('black')
            parts['cbars'].set_color('black')
            parts['cmedians'].set_linewidth(1.5)

        trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

        for center, o_pos, m_pos, group in zip(group_centers, other_positions, marker_positions, groups):
            row = results_d[(results_d['group'] == group) & (results_d['homology_dim'] == h_dim)] \
                if not results_d.empty else pd.DataFrame()
            if row.empty:
                continue
            p = row['p_empirical_adj_per_dim'].values[0]
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

            ax.plot(
                [o_pos, o_pos, m_pos, m_pos],
                [bracket_y, bracket_y + bracket_height, bracket_y + bracket_height, bracket_y],
                color='black', linewidth=1, transform=trans, clip_on=False
            )
            ax.text(center, star_y, sig, ha='center', va='bottom',
                     fontsize=STAR_FONTSIZE if sig != 'ns' else OTHER_FONTSIZE,
                     transform=trans)

        ax.set_xticks(group_centers)
        ax.set_xticklabels(groups, rotation=30, ha='right', fontsize=OTHER_FONTSIZE)
        ax.tick_params(axis='y', labelsize=OTHER_FONTSIZE)

        
        ax.set_title(f"$H_{h_dim}$", fontsize=TITLE_FONTSIZE, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel('Perturbation Score', fontsize=OTHER_FONTSIZE)

   # panel letter stays on the left, above the first column
    axes[row_idx, 0].annotate(
        panel_labels[row_idx],
        xy=(-0.25, 1.15), xycoords='axes fraction',
        fontsize=TITLE_FONTSIZE, fontweight='bold', ha='left', va='bottom'
    )

    # dataset name now centered above the middle column of this row
    axes[row_idx, mid_col].annotate(
        dataset,
        xy=(0.5, 1.15), xycoords='axes fraction',
        fontsize=TITLE_FONTSIZE, fontweight='bold', ha='center', va='bottom')

legend_elements = [
    Patch(facecolor='#3849B7', alpha=1, label='Marker'),
    Patch(facecolor='#FD6BE1', alpha=1, label='Other'),
]
legend = axes[0, -1].legend(handles=legend_elements, loc='upper right',
                             fontsize=OTHER_FONTSIZE - 2, title='Gene', bbox_to_anchor=(1, 0.85))
legend.get_title().set_fontsize(OTHER_FONTSIZE - 2)

plt.tight_layout()
plt.savefig('Thesis_Plots/score_violin_all_datasets.png', dpi=300)
plt.show()

## 9. Peak-Wise Perturbation

In [ ]:
"""
Plot selected HOMER motif logos using logomaker (from PWM in .motif files),
with proper axes, one panel per cell type.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logomaker

BASE = Path("/Users/isabelgiray/Desktop/TUM/Master_thesis")

PANELS = [
    ("PBMC",       "DC",        "PU.1",  "DC",        "PU.1 | SpiB"),
    ("PBMC",       "Monocytes",      "CEBP",  "Monocytes", "C/EBP:AP1"),
    ("HumanBrain", "Microglia", "PU.1",  "Microglia", "PU.1 | SPI1"),
]

ROW_LABELS = ["PBMC", "HumanBrain"]
PANEL_LETTERS = ["A", "B", "C"]

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
axes[1, 1].axis("off")

ax_list = [axes[0, 0], axes[0, 1], axes[1, 0]]

# row labels (vertical text, one per row)
for row_idx, row_label in enumerate(ROW_LABELS):
    fig.text(
        0.01, 0.75 - row_idx * 0.5,  # adjust y positions to center per row
        row_label,
        fontsize=14, fontweight="bold",
        va="center", ha="center",
        rotation=90,
    )

for ax, (dataset, celltype, tf, panel_label, subtitle), letter in zip(ax_list, PANELS, PANEL_LETTERS):
    homer_dir = BASE / f"top_1pct_{dataset}" / celltype
    txt_path  = homer_dir / "knownResults.txt"
    logo_dir  = homer_dir / "knownResults"

    try:
        rank       = find_motif_rank(txt_path, tf)
        motif_path = logo_dir / f"known{rank}.motif"
        pwm        = load_pwm(motif_path)

        logo = logomaker.Logo(pwm, ax=ax, color_scheme=NUC_COLORS)
        logo.style_spines(visible=False)
        logo.style_spines(spines=["left", "bottom"], visible=True)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Frequency", fontsize=14,labelpad=30)
        ax.set_xlabel("Position", fontsize=14)
        ax.tick_params(axis="both", labelsize=12)
        ax.set_xticks(range(len(pwm)))

    except Exception as e:
        ax.text(0.5, 0.5, f"Error:\n{e}", ha="center", va="center",
                transform=ax.transAxes, fontsize=8, color="red")

    ax.set_title(f"{panel_label}\n{subtitle}", fontsize=15, fontweight="bold")

    # bold black panel letter top-left
    ax.text(
        -0.18, 1.05, letter,
        transform=ax.transAxes,
        fontsize=14, fontweight="bold",
        color="black", va="top", ha="left",
    )

plt.tight_layout()
plt.subplots_adjust(left=0.08)  # make room for row labels on the left
out_path = BASE / "motif_logos_selected.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## 10. PCA Plots + Tables

In [ ]:
# nicer dataset names
dataset_titles = {
    "HumanBrain": "Human Brain",
    "MouseBrain": "Mouse Brain",
    "PBMC": "PBMC",
}

COLORS_SELECTED = [(0.34550725069638827, 0.4203708006658883, 0.9696902293486781),
 (0.9893800026041992, 0.378955911742755, 0.21756841368122667),
 (0.3959642074605608, 0.24823947872676938, 0.4016676539297192),
 (0.9937826924994482, 0.4211527500079969, 0.8812994030921271),
 (0.4140058397372807, 0.9619317608252869, 0.3109026417629064),
 (0.2286247431221609, 0.6437632542888629, 0.4081322805120583),
 (0.25003260615661993, 0.938691496932296, 0.9192515923797947),
 (0.7646511697684856, 0.24254983894398235, 0.7085129830496552),
 (0.3017721221187747, 0.6522700618245787, 0.9844707721904342),
 (0.21782892631529166, 0.2854088109905996, 0.7174819557293214),
 (0.9991896546476063, 0.4844986464266022, 0.5344476773522967),
 (0.8919236560192338, 0.7949117963224906, 0.7730486745511909),
 (0.6428145648345002, 0.31108252586505475, 0.2041098261347507),
 (0.20269999368314223, 0.7748472379028853, 0.6824940025160084),
 (0.9819923318374866, 0.6965816490867496, 0.21263663000131983),
 (0.809020033101876, 0.23056728504993104, 0.9541856467035792),
 (0.9191255237720919, 0.20589563833687236, 0.4656154484972612),
 (0.4066816185888487, 0.7781338620666193, 0.22302116197384453),
 (0.9393586762460103, 0.9614694589148117, 0.22619331869433382),
 (0.29226458048518345, 0.42830335364029093, 0.6916335500700534)]

n_rows = 3
n_cols = 2
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False
)

for i, dataset in enumerate(DATASETS):

    annot = "CellType" if dataset in ["PBMC", "HumanBrain"] else "Coarse_CellType"

    cell_types = set()
    adatas = {}

    for modality in MODALITIES:

        data = BASE / f"{dataset}/CM/{dataset}_{modality}_Def.h5ad"
        adata = sc.read_h5ad(data)

        if "X_PCA" in adata.obsm:
            adata.obsm["X_pca"] = adata.obsm.pop("X_PCA")

        adata = adata[adata.obs[annot].notna()].copy()
        adata.obs[annot] = adata.obs[annot].astype(str)

        adatas[modality] = adata
        cell_types.update(adata.obs[annot].unique())

    cell_types = sorted(cell_types)

    color_map = {
        ct: COLORS_SELECTED[j % len(COLORS_SELECTED)]
        for j, ct in enumerate(cell_types)
    }

    for j, modality in enumerate(MODALITIES):

        ax = axes[i, j]
        adata = adatas[modality]

        point_colors = [
            color_map[x]
            for x in adata.obs[annot]
        ]

        ax.scatter(
            adata.obsm["X_pca"][:, 0],
            adata.obsm["X_pca"][:, 1],
            c=point_colors,
            s=3,
            alpha=0.7,
            rasterized=True
        )

        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")

        # modality subtitles only in first row
        
        ax.set_title(
            "GEX" if modality == "GEX" else "ATAC",
            fontsize=13,
            fontweight="normal"
        )

    # dataset title centered over both panels

    # legend once per dataset
    handles = [
        mpatches.Patch(
            color=color_map[ct],
            label=ct
        )
        for ct in cell_types
    ]

    legend = axes[i, -1].legend(
    handles=handles,
    title="Cell Type",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=False,
    fontsize=12,
    title_fontsize=14
    )

plt.tight_layout()
plt.subplots_adjust(top=0.94)
dataset_titles = {
    "PBMC": "PBMC",
    "HumanBrain": "Human Brain",
    "MouseBrain": "Mouse Brain",
}

# leave room above the first row
plt.subplots_adjust(top=0.94,hspace=0.45)

for i, dataset in enumerate(DATASETS):

    left = axes[i, 0].get_position()
    right = axes[i, 1].get_position()

    # center between the two PCA plots
    x = (left.x0 + right.x1) / 2

    # a little above the row
    y = left.y1 + 0.03

    fig.text(
        x,
        y,
        dataset_titles.get(dataset, dataset),
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold",
    )
plt.savefig("PCA.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
tables = {}

for dataset in DATASETS:
    counts_per_modality = {}
    meta_per_modality = {}

    for modality in MODALITIES:
        path = ORIGIN / f"{dataset}/CM/{dataset}_{modality}_Def.h5ad"
        adata = sc.read_h5ad(path)
        annot = "CellType" if dataset in ["PBMC", "HumanBrain"] else "Coarse_CellType"

        counts_per_modality[modality] = adata.obs[annot].value_counts(dropna=False).sort_index()
        meta_per_modality[modality] = {
            "n_obs": adata.n_obs,
            "n_vars": adata.n_vars,
        }

    # cell type counts, one column per modality
    df = pd.DataFrame(counts_per_modality)
    df.index.name = "Cell Type"

    # extra summary rows, one value per modality
    feature_row = pd.Series(
        {m: meta_per_modality[m]["n_vars"] for m in MODALITIES},
        name="Number of genes/peaks",
    )
    total_row = pd.Series(
        {m: meta_per_modality[m]["n_obs"] for m in MODALITIES},
        name="Total cells",
    )

    df = pd.concat([feature_row.to_frame().T, df, total_row.to_frame().T])
    df.index.name = "Cell Type"

    # top header row spanning each modality's column
    df.columns = pd.MultiIndex.from_tuples([(m, "Count") for m in df.columns])

    tables[dataset] = df

for dataset, df in tables.items():
    latex_code = df.to_latex(
        multicolumn=True,
        multicolumn_format="c",
        caption=f"Cell type distribution for {dataset}, split by modality.",
        label=f"tab:celltypes_{dataset}",
    )
    print(latex_code)